<a href="https://colab.research.google.com/github/2410010585amel/statistika-probabilitas-uniska/blob/main/notebooks/M9_Praktikum_EDA_Mall_Customers_FadliatulAmaliyah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Modul 9 — Praktikum Statistik Deskriptif & EDA

### Reference Walkthrough: Mall Customers Dataset + Milestone 1 Template

**Mata Kuliah:** Statistika & Probabilitas (TIF1501)
**Program Studi:** Teknik Informatika (55201) · FTI · Uniska MAB Banjarmasin
**Semester:** 5 · 3 SKS · CPMK072 Dimulai

---

## 🎯 Tujuan Pembelajaran

1. **Menghitung** measures of central tendency (mean, median, modus) dan memilih yang tepat.
2. **Menghitung** measures of dispersion (variance, std, IQR, CV, range).
3. **Membuat** visualisasi univariat & bivariat untuk eksplorasi data.
4. **Menerapkan** framework EDA 7-langkah pada dataset Kaggle pilihan sendiri.

---

## 📋 Alur Notebook

| Bagian | Isi | Waktu |
|--------|-----|-------|
| **A** | Setup & Load Mall Customers | 5 menit |
| **B** | Inspeksi Awal & Data Quality | 10 menit |
| **C** | Central Tendency (mean/median/mode) | 10 menit |
| **D** | Dispersion (std/var/IQR/CV) | 10 menit |
| **E** | Univariate Visualization | 15 menit |
| **F** | Bivariate Visualization | 15 menit |
| **G** | Interactive Plotly Dashboard | 10 menit |
| **H** | Milestone 1 Template + Refleksi | 15 menit |

**Total: ± 90 menit**

---

## 💡 Mengapa Mall Customers?

- ✅ **200 pelanggan nyata** sebuah mall di Amerika
- ✅ Mix of types: 1 nominal (Gender) + 4 numeric (ID, Age, Income, Spending Score)
- ✅ Dataset klasik Kaggle untuk EDA + customer segmentation
- ✅ Insight bisnis nyata: pola belanja per gender, age-income-spending relationship
- ✅ Template langsung bisa diaplikasikan ke dataset pilihan mahasiswa

**Catatan**: Dataset ini adalah **REFERENSI**. Milestone 1 Anda harus memilih dataset Kaggle SENDIRI (≥500 rows, ≥5 kolom) dan apply framework yang sama.


---

# 🔧 A. Setup & Load Mall Customers


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Load Mall Customers dataset
url = "https://raw.githubusercontent.com/SteffiPeTaffy/machineLearningAZ/master/Machine%20Learning%20A-Z%20Template%20Folder/Part%204%20-%20Clustering/Section%2024%20-%20K-Means%20Clustering/Mall_Customers.csv"
df = pd.read_csv(url)

# Rename for easier access
df.rename(columns={
    'Genre': 'Gender',
    'Annual Income (k$)': 'Income',
    'Spending Score (1-100)': 'SpendingScore'
}, inplace=True)

print(f"Dataset berhasil dimuat!")
print(f"Shape: {df.shape[0]} baris × {df.shape[1]} kolom")
df.head()


---

# 🔍 B. Inspeksi Awal & Data Quality


In [ ]:
# Step 1: dtypes & missing values
print("=== DATA TYPES ===")
print(df.dtypes)
print("\n=== MISSING VALUES ===")
print(df.isnull().sum())
print("\n=== DUPLICATE ROWS ===")
print(f"Duplikat: {df.duplicated().sum()}")
print("\n=== MEMORY USAGE ===")
print(df.memory_usage(deep=True).sum(), "bytes")


In [ ]:
# Step 2: Statistik deskriptif ringkasan (otomatis via describe)
print("=== DESCRIBE NUMERIC ===")
df.describe().round(2)


In [ ]:
# Describe categorical (Gender)
print("=== DESCRIBE CATEGORICAL ===")
df['Gender'].value_counts()


---

# 📊 C. Measures of Central Tendency

**Tujuan Pembelajaran 1** — Mean, Median, Modus per kolom numerik.


In [ ]:
# Pilih kolom numerik (kecuali CustomerID)
numeric_cols = ['Age', 'Income', 'SpendingScore']

central = pd.DataFrame({
    'Mean': df[numeric_cols].mean().round(2),
    'Median': df[numeric_cols].median().round(2),
    'Modus': df[numeric_cols].mode().iloc[0].round(2),
    'Diff (Mean-Median)': (df[numeric_cols].mean() - df[numeric_cols].median()).round(2),
})

print("CENTRAL TENDENCY per kolom numerik:\n")
print(central)
print("\n💡 Jika Mean > Median → right-skewed")
print("   Jika Mean < Median → left-skewed")
print("   Jika Mean ≈ Median → symmetric")


In [ ]:
# Interpretasi otomatis
print("INTERPRETASI DISTRIBUSI:")
for col in numeric_cols:
    mean = df[col].mean()
    median = df[col].median()
    diff = (mean - median) / median * 100
    if abs(diff) < 2:
        shape = "SYMMETRIC (dekat Normal)"
    elif diff > 0:
        shape = f"RIGHT-SKEWED (mean {diff:.1f}% > median)"
    else:
        shape = f"LEFT-SKEWED (mean {diff:.1f}% < median)"
    print(f"  {col:<15}: {shape}")

# Modus untuk Gender (categorical)
print(f"\nModus Gender: {df['Gender'].mode()[0]} ({df['Gender'].value_counts().iloc[0]} orang)")


---

# 📏 D. Measures of Dispersion

**Tujuan Pembelajaran 2** — Variance, Std, IQR, CV.


In [ ]:
dispersion = pd.DataFrame({
    'Range': df[numeric_cols].max() - df[numeric_cols].min(),
    'Variance': df[numeric_cols].var().round(2),
    'Std Dev': df[numeric_cols].std().round(2),
    'Q1': df[numeric_cols].quantile(0.25),
    'Q3': df[numeric_cols].quantile(0.75),
    'IQR': df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25),
    'CV (%)': (df[numeric_cols].std() / df[numeric_cols].mean() * 100).round(2),
})

print("DISPERSION per kolom numerik:\n")
print(dispersion)
print("\n💡 CV tinggi = variasi relatif besar (bagus untuk bandingkan antar kolom)")
print("   IQR = robust terhadap outlier")


In [ ]:
# Interpretasi CV
print("INTERPRETASI COEFFICIENT OF VARIATION (CV):")
for col in numeric_cols:
    cv = df[col].std() / df[col].mean() * 100
    if cv < 15:
        level = "RENDAH — data cukup konsisten"
    elif cv < 30:
        level = "SEDANG — variasi wajar"
    else:
        level = "TINGGI — variasi besar antar observasi"
    print(f"  {col:<15}: CV = {cv:.1f}% → {level}")


---

# 📊 E. Univariate Visualization

**Tujuan Pembelajaran 3** — Histogram, boxplot, violin, countplot.


In [ ]:
# 2x2 grid: histogram untuk 3 numerik + countplot gender
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col, color in zip(axes.flatten()[:3], numeric_cols, ['#6366F1', '#EC4899', '#10B981']):
    df[col].hist(bins=25, ax=ax, color=color, edgecolor='black', alpha=0.8)
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean={df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='orange', linestyle='--', linewidth=2, label=f'Median={df[col].median():.1f}')
    ax.set_title(f'Distribusi {col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.legend()

# Gender countplot
ax = axes[1, 1]
counts = df['Gender'].value_counts()
bars = ax.bar(counts.index, counts.values, color=['#F59E0B', '#6366F1'], edgecolor='black', alpha=0.85)
ax.set_title('Distribusi Gender', fontweight='bold')
ax.set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val} ({val/len(df)*100:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Boxplot untuk deteksi outlier
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, color in zip(axes, numeric_cols, ['#6366F1', '#EC4899', '#10B981']):
    bp = ax.boxplot(df[col], patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.5)
    ax.set_title(f'Boxplot {col}', fontweight='bold')
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

# Deteksi outlier via IQR
print("\n=== DETEKSI OUTLIER (IQR method) ===")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"  {col:<15}: {len(outliers)} outlier (<{Q1-1.5*IQR:.1f} atau >{Q3+1.5*IQR:.1f})")


In [ ]:
# Violin plot — distribusi + density
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, color in zip(axes, numeric_cols, ['#6366F1', '#EC4899', '#10B981']):
    parts = ax.violinplot(df[col], widths=0.6, showmeans=True, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.6)
    ax.set_title(f'Violin {col}', fontweight='bold')
    ax.set_ylabel(col)
    ax.set_xticks([1])
    ax.set_xticklabels([col])

plt.tight_layout()
plt.show()


---

# 🔗 F. Bivariate Visualization

**Tujuan Pembelajaran 3** — Scatter, correlation heatmap, pair plot, grouped boxplot.


In [ ]:
# Scatter: Income vs SpendingScore (dengan warna Gender)
fig, ax = plt.subplots(figsize=(11, 7))
for gender, color in [('Male', '#6366F1'), ('Female', '#EC4899')]:
    sub = df[df['Gender'] == gender]
    ax.scatter(sub['Income'], sub['SpendingScore'], c=color, label=gender, s=80, alpha=0.7, edgecolors='black')

ax.set_xlabel('Annual Income (k$)', fontsize=12)
ax.set_ylabel('Spending Score (1-100)', fontsize=12)
ax.set_title('Income vs Spending Score — warna per Gender', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("\n💡 Visual terlihat 5 cluster alami — ini landasan customer segmentation!")


In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[numeric_cols + ['CustomerID']].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5, square=True)
ax.set_title('Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print("\n💡 PENTING: CustomerID korelasi 0.978 dengan Income — ini artifact karena data diurutkan!")
print("   Drop CustomerID sebelum analisis lanjut.")


In [ ]:
# Pair plot — scatter matrix semua pasangan numerik
g = sns.pairplot(df[numeric_cols + ['Gender']], hue='Gender',
                 palette={'Male': '#6366F1', 'Female': '#EC4899'},
                 diag_kind='kde', height=2.8)
g.fig.suptitle('Pair Plot: Mall Customers (hue=Gender)', y=1.02, fontsize=14, fontweight='bold')
plt.show()


In [ ]:
# Grouped boxplot: bandingkan metric per Gender
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(x='Gender', y=col, data=df, ax=ax,
                palette={'Male': '#6366F1', 'Female': '#EC4899'})
    ax.set_title(f'{col} per Gender', fontweight='bold')

plt.tight_layout()
plt.show()

# Statistical comparison
print("\n=== PERBANDINGAN MEAN per GENDER ===")
print(df.groupby('Gender')[numeric_cols].mean().round(2))


---

# 📊 G. Interactive Plotly Dashboard

**Tujuan Pembelajaran 4** — Dashboard interactive siap-share.


In [ ]:
# Interactive scatter dengan hover info
fig = px.scatter(df, x='Income', y='SpendingScore',
                 color='Gender', size='Age',
                 hover_data=['CustomerID', 'Age'],
                 title='Mall Customers: Income vs Spending Score (interaktif)',
                 color_discrete_map={'Male': '#6366F1', 'Female': '#EC4899'},
                 size_max=25)
fig.update_layout(width=900, height=500, font=dict(size=12))
fig.show()


In [ ]:
# Multi-panel dashboard
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=('Distribusi Age', 'Distribusi Income',
                                    'Distribusi Spending Score', 'Gender Breakdown'),
                    specs=[[{'type':'histogram'}, {'type':'histogram'}],
                           [{'type':'histogram'}, {'type':'pie'}]])

fig.add_trace(go.Histogram(x=df['Age'], marker_color='#6366F1', nbinsx=25), row=1, col=1)
fig.add_trace(go.Histogram(x=df['Income'], marker_color='#EC4899', nbinsx=25), row=1, col=2)
fig.add_trace(go.Histogram(x=df['SpendingScore'], marker_color='#10B981', nbinsx=25), row=2, col=1)

counts = df['Gender'].value_counts()
fig.add_trace(go.Pie(labels=counts.index, values=counts.values,
                     marker=dict(colors=['#EC4899', '#6366F1'])), row=2, col=2)

fig.update_layout(height=700, width=1000, showlegend=False,
                  title_text='Mall Customers Interactive Dashboard',
                  title_font_size=16)
fig.show()


---

# 📝 H. Milestone 1 Template + Refleksi

## H.1 Insight dari Mall Customers (Reference)

Dari EDA di atas, berikut 5 insight bisnis yang bisa diambil:

1. **Gender Balance**: 56% Female, 44% Male — sedikit skew ke perempuan.
2. **Age Distribusi**: slight right-skewed (mean 38.9 > median 36), rentang 18-70 tahun.
3. **Income vs Spending TIDAK ada korelasi linier** (r ≈ 0.01) — insight kunci untuk segmentasi.
4. **5 cluster alami terlihat di scatter Income vs Spending** → landasan K-Means (M12+).
5. **Spending Score simetris** (mean=median=50) — mungkin skor dirancang dengan normalisasi.

## H.2 Template Milestone 1 untuk Dataset Anda

Setelah menyelesaikan reference di atas, buat notebook baru untuk dataset pilihan Anda:


> ℹ️ **Catatan output:** notebook ini mengunduh data dari internet, jadi output sel **belum ada** di file ini. Jalankan **Runtime → Run all** di Google Colab untuk menghasilkan output. Angka bertanda **≈** pada jawaban di bawah adalah nilai yang diharapkan dari statistik standar dataset; cocokkan dengan output Anda.

## H.2b Contoh Pengisian Template Milestone 1

Cell di bawah adalah contoh template yang sudah diisi memakai dataset *Students Performance in Exams*. Ganti dengan dataset pilihan Anda jika berbeda (≥ 500 baris, ≥ 5 kolom, bukan Mall Customers).

In [ ]:
# MILESTONE 1 (contoh pengisian) — Students Performance in Exams
# Unduh dataset dari Kaggle (butuh kaggle.json), lalu jalankan 7 langkah EDA.
# !kaggle datasets download -d spscientist/students-performance-in-exams
# !unzip -o students-performance-in-exams.zip

import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
df_my = pd.read_csv('StudentsPerformance.csv')

# STEP 1: Load & inspeksi
print(f"Shape: {df_my.shape}"); display(df_my.head())

# STEP 2: Missing & duplikat
print(df_my.isnull().sum()); print(f"Duplicates: {df_my.duplicated().sum()}")

# STEP 3: Central tendency
numeric = df_my.select_dtypes(include='number').columns.tolist()
print(df_my[numeric].describe())

# STEP 4: Dispersion
print({c: {'std': df_my[c].std(), 'IQR': df_my[c].quantile(0.75) - df_my[c].quantile(0.25),
           'CV': df_my[c].std() / df_my[c].mean() * 100} for c in numeric})

# STEP 5: Univariate (3 grafik)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, c in zip(ax, numeric): sns.histplot(df_my[c], kde=True, ax=a); a.set_title(c)
plt.show()

# STEP 6: Bivariate (scatter + heatmap + boxplot per kelompok)
sns.scatterplot(data=df_my, x=numeric[1], y=numeric[2]); plt.show()
sns.heatmap(df_my[numeric].corr(), annot=True, cmap='RdBu_r', center=0); plt.show()
sns.boxplot(data=df_my, x='test preparation course', y=numeric[0]); plt.show()

# STEP 7: tulis 3-5 insight (markdown) berdasarkan angka di atas


In [ ]:
# =========================================================
# TEMPLATE MILESTONE 1 — ganti URL dengan dataset Anda
# =========================================================

# STEP 1: Load Dataset
# URL = "<URL raw CSV dataset Kaggle Anda>"
# df_my = pd.read_csv(URL)
# print(f"Shape: {df_my.shape}")
# df_my.head()

# STEP 2: Missing & Duplicate Check
# print(df_my.isnull().sum())
# print(f"Duplicates: {df_my.duplicated().sum()}")

# STEP 3: Central Tendency
# numeric = df_my.select_dtypes(include='number').columns.tolist()
# print(df_my[numeric].describe())

# STEP 4: Dispersion
# print({c: {'std': df_my[c].std(), 'IQR': df_my[c].quantile(0.75) - df_my[c].quantile(0.25),
#            'CV': df_my[c].std() / df_my[c].mean() * 100} for c in numeric})

# STEP 5: Univariate Viz — min 3 histogram/boxplot
# (contoh kode ada di cell sebelumnya, adaptasi untuk kolom Anda)

# STEP 6: Bivariate Viz — min scatter + heatmap
# sns.heatmap(df_my[numeric].corr(), annot=True, cmap='RdBu_r', center=0)

# STEP 7: 3-5 Insight tertulis (markdown cell)
# - Insight 1: ...
# - Insight 2: ...
# - Insight 3: ...

print("💡 Copy template ini ke notebook baru, ganti URL dan nama kolom.")
print("   Submit sebelum Modul 10 dimulai (link GitHub via LMS).")


## H.3 Pertanyaan Refleksi (sudah dijawab)

### ✏️ Jawaban Anda:

**1. Mean 60,6 vs median 61,5 pada Income.** Hampir sama, artinya distribusi **simetris** (tidak miring) dan tidak punya outlier besar yang menarik rata-rata (selisih hanya −1,5%). Jadi mean boleh dipakai sebagai ukuran pemusatan, dan median tetap aman sebagai pembanding. Kalau keduanya berjauhan, pakai median.

**2. Contoh spurious correlation lain.** Penjualan es krim naik bersama jumlah kasus orang tenggelam. Keduanya dipicu **musim panas**, bukan saling menyebabkan. Bahayanya: keputusan salah (misal melarang penjualan es krim). Dalam ML, fitur seperti ID atau tanggal membuat model "belajar" pola palsu yang tidak berlaku pada data baru.

**3. Kapan IQR dipilih dibanding Std Dev?** Saat data **miring** atau punya **outlier**, misalnya gaji, harga rumah, atau waktu respons API. Satu CEO atau satu transaksi ekstrem menarik std sangat besar, sementara IQR (50% data tengah) tetap stabil. Std cocok untuk data simetris tanpa outlier.

**4. Strategi bisnis jika ada 5 segmen.** Beri kampanye berbeda untuk tiap segmen. Contoh: **pendapatan tinggi tetapi belanja rendah** → program loyalitas, promosi premium, atau acara eksklusif agar mereka lebih sering belanja. **Pendapatan rendah tetapi belanja tinggi** → diskon anggota dan cicilan. **Segmen tengah** → promosi umum.

**5. Dataset Milestone 1 (contoh, silakan ganti dengan pilihan Anda sendiri).**
- **Dataset:** *Students Performance in Exams* (Kaggle, 1.000 baris, 8 kolom: gender, race/ethnicity, parental level of education, lunch, test preparation course, math/reading/writing score).
- **Konteks bisnis:** institusi pendidikan ingin tahu faktor yang berhubungan dengan nilai ujian.
- **3 pertanyaan EDA:** (a) Apakah siswa yang ikut kursus persiapan punya nilai rata-rata lebih tinggi? (b) Apakah tingkat pendidikan orang tua berkaitan dengan nilai? (c) Seberapa kuat hubungan antara nilai matematika, membaca, dan menulis?
- **Hipotesis untuk Milestone 2:** H₁: nilai rata-rata siswa dengan kursus persiapan lebih tinggi. H₂: ada perbedaan nilai antar tingkat pendidikan orang tua. H₃: nilai membaca dan menulis berkorelasi positif kuat.


## H.4 Checklist Submit Milestone 1

### Reference Walkthrough (notebook ini):
- [ ] Semua cell sukses dijalankan tanpa error
- [ ] Central tendency table muncul lengkap
- [ ] Dispersion table muncul lengkap
- [ ] Histogram + boxplot + violin + countplot muncul
- [ ] Scatter + heatmap + pairplot muncul
- [ ] Plotly dashboard interaktif
- [ ] 5 pertanyaan refleksi dijawab

### Milestone 1 Deliverable (notebook SENDIRI):
- [ ] Dataset Kaggle pilihan (≥500 rows, ≥5 kolom, bukan Mall Customers)
- [ ] Framework EDA 7-langkah lengkap
- [ ] Minimal 6 visualisasi (3 univariat + 3 bivariat)
- [ ] 1 interactive Plotly dashboard (minimal 2 chart)
- [ ] 3-5 insight tertulis dengan evidence
- [ ] Daftar 2-3 hipotesis untuk Milestone 2 (M10 hypothesis testing)
- [ ] README.md dengan: deskripsi dataset, sumber, cara menjalankan, ringkasan insight
- [ ] Repo GitHub ter-commit dan link dikumpulkan via LMS

### Deadline: SEBELUM pertemuan Modul 10 dimulai.

## H.5 Push ke GitHub

1. Buat repository baru: `projek-statistika-<nama>` di GitHub
2. Upload notebook, README, dan screenshot dashboard
3. Commit message: `"Milestone 1: EDA dataset <nama-dataset>"`
4. Submit link repo via LMS Uniska

---

## 🎓 Selamat! Anda Berhasil...

✅ Menghitung semua measures of central tendency dan dispersion
✅ Membuat 6 jenis visualisasi berbeda
✅ Membangun interactive Plotly dashboard
✅ Mengekstrak 5 insight bisnis dari dataset
✅ Siap aplikasi ke Milestone 1 dengan dataset pilihan sendiri

## 📚 Bacaan Lanjutan

- **Bruce, P.** (2020). *Practical Statistics for Data Scientists*, Chapter 1 (Exploratory Data Analysis).
- **Tukey, J.** (1977). *Exploratory Data Analysis* — pionir boxplot & five-number summary.
- **Plotly Documentation** — interactive dashboards: https://plotly.com/python/
- **Seaborn Gallery** — https://seaborn.pydata.org/examples/

## 🔮 Persiapan Modul 10

Modul 10 — **Inferential Statistics & Hypothesis Testing**:
- Confidence Interval (CI 95%)
- t-test, chi-square, ANOVA
- p-value interpretation
- **Milestone 2**: Uji 2-3 hipotesis yang Anda formulasikan di akhir M9

**Sebelum M10:**
1. Selesaikan Milestone 1 EDA, push ke GitHub
2. Formulasikan 3 hipotesis dari insight Milestone 1 Anda
3. Baca Bruce Ch. 3 (Statistical Experiments and Significance Testing)

---

*Notebook by: Prodi Teknik Informatika · Uniska MAB Banjarmasin · Edisi 2026*
